In [ ]:
%pip install ydata-profiling
import pandas as pd
from ydata_profiling import ProfileReport

# Temizlenmiş veriyi okuyoruz
df_oyunlar = pd.read_csv("../../data/processed/oyunlar_temiz.csv")

# 1. AGRESİF KALKAN: Sadece oyun adını değil, binlerce farklı metin içeren geliştiriciyi de çıkarıyoruz.
df_eda = df_oyunlar.drop(columns=['oyun_adi', 'gelistirici'], errors='ignore')

# 2. HIZLI MOD: minimal=True ile devam
profil_raporu = ProfileReport(df_eda, title="Oyun Verileri EDA Raporu", minimal=True)

# 3. KİLİTLENME ÇÖZÜMÜ: to_widgets() komutunu TAMAMEN siliyoruz! 
# Notebook içinde göstermeye çalışıp VS Code'u boğmayacağız.

# Doğrudan reports klasörüne kaydediyoruz
rapor_yolu = "../../reports/oyunlar_eda_raporu.html"
profil_raporu.to_file(rapor_yolu)

print(f"✅ Rapor oluşturma işlemi bitti!")
print(f"Lütfen sol taraftaki dosya yöneticisinden '{rapor_yolu}' dosyasını bulup normal bir tarayıcıda (Chrome, Edge vb.) açın.")

Note: you may need to restart the kernel to use updated packages.


c:\Users\betul\Notes\ANIVIA\ybs-makine-ogrenmesi\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\betul\AppData\Local\Temp\ipykernel_28412\478739229.py:3: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport
Export report to file: 100%|██████████| 1/1 [00:00<00:00, 169.26it/s]

✅ Rapor oluşturma işlemi bitti!
Lütfen sol taraftaki dosya yöneticisinden '../../reports/oyunlar_eda_raporu.html' dosyasını bulup normal bir tarayıcıda (Chrome, Edge vb.) açın.


In [ ]:
# ---------------------------------------------------------
# Veriyi Çekme ve Temizleme
# ---------------------------------------------------------

import pandas as pd
import plotly.express as px
from IPython.display import display

# 1. Veriyi okuyoruz
dosya_yolu = "../../data/processed/oyunlar_temiz.csv"
df_oyunlar = pd.read_csv(dosya_yolu)

# 2. Kategori sütunundaki değerleri birleştir (.str.replace kullanarak)
df_oyunlar['kategori'] = df_oyunlar['kategori'].str.replace('Action', 'Aksiyon', regex=False)
df_oyunlar['kategori'] = df_oyunlar['kategori'].str.replace('Adventure', 'Macera', regex=False)

# 3. Orijinal oyun adı sütununu grafikleri boğmaması için EDA öncesi geçici olarak siliyoruz
df_eda_icin = df_oyunlar.drop(columns=['oyun_adi'], errors='ignore')

def plotly_otomatik_eda(df):
    print("="*60)
    print("📊 VERİ SETİ GENEL BAKIŞ (SUMMARY)")
    print("="*60)
    print(f"Toplam Satır: {df.shape[0]}")
    print(f"Toplam Sütun: {df.shape[1]}")
    
    # Eksik Veri Analizi Tablosu
    eksik_df = pd.DataFrame({
        'Eksik Değer Sayısı': df.isnull().sum(),
        'Eksik Oranı (%)': (df.isnull().sum() / len(df) * 100).round(2)
    })
    print("\nEksik Veri (NaN) Analizi:")
    display(eksik_df[eksik_df['Eksik Değer Sayısı'] > 0].sort_values(by='Eksik Oranı (%)', ascending=False))
    
    print("\n" + "="*60)
    print("📈 SÜTUN BAZLI GÖRSEL DAĞILIMLAR")
    print("="*60)

    for col in df.columns:
        s_valid = df[col].dropna()
        if s_valid.empty:
            continue

        nunique = s_valid.nunique()

        # 1. KATEGORİK VEYA METİN VERİLER (Platform, Cihaz, Kategori, Geliştirici)
        if df[col].dtype == 'object' or str(df[col].dtype) == 'category' or nunique <= 20:
            if nunique > 15:
                # Kilitlenmeyi önlemek için sadece en çok geçen 15 değeri al
                title = f"'{col}' Dağılımı (En Çok Geçen İlk 15) - Toplam {nunique} Farklı Değer"
                val_counts = s_valid.value_counts().head(15).reset_index()
            else:
                title = f"'{col}' Dağılımı - Toplam {nunique} Farklı Değer"
                val_counts = s_valid.value_counts().reset_index()

            val_counts.columns = [col, 'Frekans']

            # "cihaz_turu" için özel renk paletini kullan
            if col == 'cihaz_turu':
                # PC ve Konsol için kontrastlı renkler
                color_map = {
                    'PC': '#1f77b4',       # Mavi
                    'Konsol': '#ff7f0e',   # Turuncu
                    'Tablet': '#2ca02c',   # Yeşil
                    'Telefon': '#d62728'   # Kırmızı
                }
                fig = px.bar(val_counts, x=col, y='Frekans', title=title, 
                             text='Frekans', color=col, color_discrete_map=color_map)
            else:
                # Diğer sütunlar için logo renk paletini kullan
                fig = px.bar(val_counts, x=col, y='Frekans', title=title, 
                             text='Frekans', color='Frekans', color_continuous_scale=['#FFD700', '#D4AF37', '#00A3E0', '#004990'])
            
            fig.update_traces(textposition='outside')
            fig.update_layout(xaxis_tickangle=-45, height=450)
            fig.show()

        # 2. SAYISAL VERİLER (Fiyat, İndirilme Sayısı)
        elif pd.api.types.is_numeric_dtype(df[col]):
            title = f"'{col}' Sayısal Dağılımı ve Aykırı Değerler"
            # Histogram ve üstüne Kutu Grafiği (Boxplot) eklenmiş harika görünüm
            fig = px.histogram(df, x=col, title=title, marginal="box", 
                               nbins=50, color_discrete_sequence=['#004990'])
            fig.update_layout(height=450)
            fig.show()

# 4. Kendi yazdığımız fonksiyonu ateşliyoruz!
plotly_otomatik_eda(df_eda_icin)

📊 VERİ SETİ GENEL BAKIŞ (SUMMARY)
Toplam Satır: 20336
Toplam Sütun: 8

Eksik Veri (NaN) Analizi:


,Eksik Değer Sayısı,Eksik Oranı (%)
fiyat,1672,8.22



📈 SÜTUN BAZLI GÖRSEL DAĞILIMLAR


In [ ]:
# Tek Pasta Grafiği: Oyun Güvenliği Analizi

# İstatistikleri hesaplıyoruz
toplam_oyun = len(df_oyunlar)
tehlikeli_oyun = (df_oyunlar['tehlikeli_oyun'] == 1).sum()
tehlikeli_kategori = (df_oyunlar['tehlikeli_kategorisi'] == 1).sum()
guvenli_oyunlar = toplam_oyun - tehlikeli_oyun - tehlikeli_kategori

# Pasta grafiği için veri hazırlama
pasta_veri = pd.DataFrame({
    'Kategori': ['Tehlikeli Oyun Sayısı', 'Tehlikeli Kategori Sayısı', 'Güvenli Oyunlar'],
    'Sayı': [tehlikeli_oyun, tehlikeli_kategori, guvenli_oyunlar]
})

# Pie chart oluştur
fig = px.pie(pasta_veri, 
             values='Sayı', 
             names='Kategori',
             title='Toplam Oyun Sayısı: Güvenlik Durumu Analizi')

# Logo renklerini manuel olarak ata
# Sıra: Tehlikeli Oyun Sayısı, Tehlikeli Kategori Sayısı, Güvenli Oyunlar
fig.update_traces(marker=dict(colors=['#FFD700', '#D4AF37', '#004990']),
                  textposition='inside', 
                  textinfo='percent+label')

fig.update_layout(height=550)
fig.show()

# İstatistikleri metin olarak da gösterelim
print("\n" + "="*60)
print("📊 OYUN GÜVENLİĞİ ANALİZİ")
print("="*60)
print(f"Toplam Oyun Sayısı: {toplam_oyun}")
print(f"\n🎮 DAĞILIM:")
print(f"  Tehlikeli Oyun Sayısı: {tehlikeli_oyun} ({tehlikeli_oyun/toplam_oyun*100:.2f}%) - Parlak Altın (#FFD700)")
print(f"  Tehlikeli Kategori Sayısı: {tehlikeli_kategori} ({tehlikeli_kategori/toplam_oyun*100:.2f}%) - Altın (#D4AF37)")
print(f"  Güvenli Oyunlar: {guvenli_oyunlar} ({guvenli_oyunlar/toplam_oyun*100:.2f}%) - Koyu Mavi (#004990)")
print("="*60)

KeyError: 'tehlikeli_kategorisi'

In [ ]:
# PEGI (Yaş Sınırı) vs Tehlikeli Kategori Analizi

from scipy.stats import chi2_contingency
import numpy as np

print("\n" + "="*70)
print("🔍 PEGI (YAŞ SINIRI) vs TEHLİKELİ KATEGORİ ÇAKIŞMA ANALİZİ")
print("="*70)

# 1. Cross-Tabulation (Çapraz Tablo)
crosstab = pd.crosstab(df_oyunlar['yas_siniri'], df_oyunlar['tehlikeli_kategorisi'], margins=True)
print("\n📊 ÇAPRAZ TABLO (Yaş Sınırı x Tehlikeli Kategori):")
print(crosstab)

# 2. Yüzde Hesaplaması
print("\n📈 TEHLİKELİ KATEGORİ ORANI (Her Yaş Sınırı için):")
crosstab_pct = pd.crosstab(df_oyunlar['yas_siniri'], df_oyunlar['tehlikeli_kategorisi'], normalize='index') * 100
print(crosstab_pct.round(2))

# 3. Chi-Square Testi
chi2, p_value, dof, expected = chi2_contingency(pd.crosstab(df_oyunlar['yas_siniri'], df_oyunlar['tehlikeli_kategorisi']))
print(f"\n🧮 CHI-SQUARE TESTİ:")
print(f"  Chi-Square Değeri: {chi2:.2f}")
print(f"  P-Value: {p_value:.2e}")
print(f"  Serbestlik Derecesi: {dof}")

# 4. İstatistiksel Anlamlılık
if p_value < 0.05:
    print(f"  ✅ İstatistiksel olarak ANLAMLI ilişki vardır (p < 0.05)")
else:
    print(f"  ❌ İstatistiksel olarak ANLAMLI ilişki YOKTUR (p >= 0.05)")

# 5. Detaylı Yorum
print("\n" + "="*70)
print("📋 YORUM VE TAVSIYE:")
print("="*70)

tehlikeli_yas_dist = crosstab_pct[1]  # Tehlikeli kategori oranları
print(f"\nTehlikeli Kategori Oranları:")
for age, pct in tehlikeli_yas_dist.items():
    print(f"  PEGI {age}: %{pct:.1f}")

# Oran varyasyonunu hesapla
oran_fark = tehlikeli_yas_dist.max() - tehlikeli_yas_dist.min()
print(f"\nMaksimum Oran Farkı: %{oran_fark:.1f}")

if oran_fark > 15:
    print("✅ SONUÇ: PEGI yaş sınırı ile tehlikeli kategori arasında GÜÇLÜ ilişki var!")
    print("   → Clustering ihtiyaç VAR. Model için önemli feature olabilir.")
elif oran_fark > 5:
    print("⚠️  SONUÇ: PEGI yaş sınırı ile tehlikeli kategori arasında ORTA ilişki var!")
    print("   → Clustering yapabilir, fakat daha fazla feature ile kombine etmelisin.")
else:
    print("❌ SONUÇ: PEGI yaş sınırı ile tehlikeli kategori arasında ZAYIF ilişki var!")
    print("   → Clustering yapmadan önce başka feature kombinasyonlarını incele.")

print("="*70)


🔍 PEGI (YAŞ SINIRI) vs TEHLİKELİ KATEGORİ ÇAKIŞMA ANALİZİ

📊 ÇAPRAZ TABLO (Yaş Sınırı x Tehlikeli Kategori):
tehlikeli_kategorisi      0     1    All
yas_siniri                              
0                      1374  2043   3417
3                      5417   287   5704
4                       137     0    137
6                         1     0      1
7                      1950   407   2357
9                       109     0    109
10                      856    45    901
12                     3038   811   3849
13                      120     0    120
15                        2     2      4
16                     1284   812   2096
18                     1055   586   1641
All                   15343  4993  20336

📈 TEHLİKELİ KATEGORİ ORANI (Her Yaş Sınırı için):
tehlikeli_kategorisi       0      1
yas_siniri                         
0                      40.21  59.79
3                      94.97   5.03
4                     100.00   0.00
6                     100.00   0.00
7       

In [ ]:
# Grouped Bar Chart: PEGI vs Tehlikeli Kategori

print("\n" + "="*70)
print("📊 GROUPED BAR CHART: PEGI YAŞ SINIRI vs TEHLİKELİ KATEGORİ")
print("="*70)

# 1. Çapraz tablo oluştur (margins olmadan)
crosstab_viz = pd.crosstab(df_oyunlar['yas_siniri'], df_oyunlar['tehlikeli_kategorisi'])

# 2. Long format'a dönüştür (Plotly Express için)
crosstab_long = crosstab_viz.reset_index()
crosstab_long = crosstab_long.melt(id_vars='yas_siniri', 
                                     var_name='tehlikeli_kategorisi', 
                                     value_name='Sayı')

# 3. Tehlikeli kategori sütununu string olarak güvenli isimlendirme
crosstab_long['Güvenlik Durumu'] = crosstab_long['tehlikeli_kategorisi'].map({
    0: 'Güvenli',
    1: 'Tehlikeli'
})

print(f"✓ Veri hazırlandı: {len(crosstab_long)} kayıt")

# 4. Grouped Bar Chart oluştur
fig_grouped = px.bar(
    crosstab_long,
    x='yas_siniri',
    y='Sayı',
    color='Güvenlik Durumu',
    barmode='group',
    title='PEGI Yaş Sınırı ve Tehlikeli Kategori Dağılımı',
    color_discrete_map={
        'Güvenli': '#004990',
        'Tehlikeli': '#FFD700'
    },
    labels={'yas_siniri': 'Yaş Sınırı (PEGI)', 'Sayı': 'Oyun Sayısı'},
    text='Sayı',
    height=550
)

# 5. Tasarım iyileştirmeleri
fig_grouped.update_traces(textposition='outside', textfont=dict(size=10))
fig_grouped.update_layout(
    hovermode='x unified',
    legend=dict(
        title='Güvenlik Durumu',
        yanchor='top',
        y=0.99,
        xanchor='right',
        x=0.99
    ),
    xaxis=dict(type='category')
)

fig_grouped.show()

print("✅ Grouped bar chart başarıyla oluşturuldu!")
print("="*70)